# Travel Analytics MLOps — Colab runner
**GitHub repository:** https://github.com/St-virus1210/travel-analytics-capstone

Upload the complete project ZIP, then run all cells. The notebook trains models, verifies artifacts, runs pytest, and performs in-process API smoke tests. Docker, Airflow, Jenkins and Kubernetes still need their own environments and real screenshots.

In [ ]:
from google.colab import files
uploaded=files.upload()
zip_names=[n for n in uploaded if n.lower().endswith('.zip')]
assert zip_names, 'Upload the complete travel-analytics-capstone.zip'
ZIP_PATH=zip_names[0]


In [ ]:
from pathlib import Path
import zipfile, shutil, os, sys
WORK=Path('/content/travel-analytics-capstone'); unpack=Path('/content/unpacked-project')
if WORK.exists(): shutil.rmtree(WORK)
if unpack.exists(): shutil.rmtree(unpack)
with zipfile.ZipFile(ZIP_PATH) as z: z.extractall(unpack)
candidate=unpack/'travel-analytics-capstone'
assert candidate.exists(), 'ZIP should contain travel-analytics-capstone/ at its root'
shutil.move(str(candidate),str(WORK)); os.chdir(WORK); sys.path.insert(0,str(WORK/'src'))
print('Project root:',WORK)


In [ ]:
%pip install -q -r /content/travel-analytics-capstone/requirements.txt
%pip install -q pytest


In [ ]:
import pandas as pd
for n in ['flights','hotels','users']:
 df=pd.read_csv(WORK/'data'/f'{n}.csv'); print(n,df.shape); display(df.head(3))


In [ ]:
import json
from travel_analytics.train import train
os.environ['MLFLOW_TRACKING_URI']=(WORK/'mlruns').as_uri()
results=train(data_dir=str(WORK/'data'),artifact_dir=str(WORK/'artifacts'))
print(json.dumps(results,indent=2))


In [ ]:
%cd /content/travel-analytics-capstone
!PYTHONPATH=src pytest -q
!PYTHONPATH=src python scripts/validate_project.py --require-artifacts


In [ ]:
# In-process API checks; no public tunnel or server required.
from travel_analytics.api import create_app
app=create_app(model_dir=WORK/'artifacts',data_dir=WORK/'data'); client=app.test_client()
health=client.get('/health'); print('health',health.status_code,health.get_json()); assert health.status_code==200
row=pd.read_csv(WORK/'data'/'flights.csv').iloc[0]
payload={'from':str(row['from']),'to':str(row['to']),'flightType':str(row['flightType']),'time':float(row['time']),'distance':float(row['distance']),'agency':str(row['agency']),'date':str(row['date'])}
response=client.post('/predict/flight-price',json=payload); print('prediction',response.status_code,response.get_json()); assert response.status_code==200
response=client.get('/recommendations?limit=5'); print('recommendations',response.status_code,response.get_json()); assert response.status_code==200


## External evidence still required

Capture screenshots only after you actually run the API/UI, Docker Compose, MLflow, Airflow, Jenkins and Kubernetes. This notebook does not pretend to start remote services.